In [1]:
import pandas as pd
df_clean = pd.read_pickle('../data/cleaned_loans.pkl')
print(df_clean.shape)

(1343086, 92)


In [2]:
'dti' in df_clean.columns

True

In [3]:
print(df_clean['dti'].describe())

count    1.343086e+06
mean     1.828518e+01
std      1.115266e+01
min     -1.000000e+00
25%      1.180000e+01
50%      1.762000e+01
75%      2.405000e+01
max      9.990000e+02
Name: dti, dtype: float64


In [4]:
print((df_clean['dti'] < 0).sum())
print((df_clean['dti'] > 100).sum())

2
531


In [5]:
before = df_clean.shape[0]
df_clean = df_clean[(df_clean['dti'] >= 0) & (df_clean['dti'] <= 100)]
after = df_clean.shape[0]
print(f"Dropped {before - after} rows. New shape: {df_clean.shape}")

Dropped 533 rows. New shape: (1342553, 92)


In [6]:
print(df_clean[['issue_d', 'earliest_cr_line']].head())
print(df_clean['issue_d'].dtype)

    issue_d earliest_cr_line
0  Dec-2015         Aug-2003
1  Dec-2015         Dec-1999
2  Dec-2015         Aug-2000
4  Dec-2015         Jun-1998
5  Dec-2015         Oct-1987
object


In [7]:
df_clean['issue_d'] = pd.to_datetime(df_clean['issue_d'], format='%b-%Y')
df_clean['earliest_cr_line'] = pd.to_datetime(df_clean['earliest_cr_line'], format='%b-%Y')

In [8]:
df_clean['credit_hist_length_months'] = (
    (df_clean['issue_d'] - df_clean['earliest_cr_line']).dt.days / 30.44
).round(0).astype(int)

print(df_clean['credit_hist_length_months'].describe())

count    1.342553e+06
mean     1.950990e+02
std      9.007208e+01
min      1.200000e+01
25%      1.350000e+02
50%      1.770000e+02
75%      2.400000e+02
max      9.990000e+02
Name: credit_hist_length_months, dtype: float64


In [9]:
print((df_clean['credit_hist_length_months'] >= 900).sum())

1


In [10]:
before = df_clean.shape[0]
df_clean = df_clean[df_clean['credit_hist_length_months'] < 900]
after = df_clean.shape[0]
print(f"Dropped {before - after} row. New shape: {df_clean.shape}")

print(df_clean['credit_hist_length_months'].describe())

Dropped 1 row. New shape: (1342552, 93)
count    1.342552e+06
mean     1.950984e+02
std      9.006944e+01
min      1.200000e+01
25%      1.350000e+02
50%      1.770000e+02
75%      2.400000e+02
max      8.520000e+02
Name: credit_hist_length_months, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

feature_cols = ['dti', 'credit_hist_length_months']  # we'll expand this list shortly
X = df_clean[feature_cols]
y = df_clean['default']  # adjust if your target column has a different name

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

(1074041, 2) (268511, 2)
default
0    0.801667
1    0.198333
Name: proportion, dtype: float64


In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(X_train_balanced.shape)
print(y_train_balanced.value_counts(normalize=True))

(1722046, 2)
default
0    0.5
1    0.5
Name: proportion, dtype: float64


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_balanced, y_train_balanced)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred_proba))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

AUC: 0.5824115085888375
Accuracy: 0.570289485346969
              precision    recall  f1-score   support

           0       0.84      0.58      0.68    215256
           1       0.24      0.54      0.33     53255

    accuracy                           0.57    268511
   macro avg       0.54      0.56      0.51    268511
weighted avg       0.72      0.57      0.61    268511



In [14]:
print(df_clean[['int_rate', 'annual_inc', 'loan_amnt']].describe())

           int_rate    annual_inc     loan_amnt
count  1.342552e+06  1.342552e+06  1.342552e+06
mean   1.323408e+01  7.628578e+04  1.441186e+04
std    4.760988e+00  6.986960e+04  8.709939e+03
min    5.310000e+00  4.300000e+02  5.000000e+02
25%    9.750000e+00  4.600000e+04  8.000000e+03
50%    1.274000e+01  6.500000e+04  1.200000e+04
75%    1.599000e+01  9.000000e+04  2.000000e+04
max    3.099000e+01  1.099920e+07  4.000000e+04


In [15]:
print((df_clean['annual_inc'] > 500000).sum())
print((df_clean['annual_inc'] > 1000000).sum())
import numpy as np
df_clean['log_annual_inc'] = np.log1p(df_clean['annual_inc'])
print(df_clean['log_annual_inc'].describe())

1742
287
count    1.342552e+06
mean     1.108587e+01
std      5.374687e-01
min      6.066108e+00
25%      1.073642e+01
50%      1.108216e+01
75%      1.140758e+01
max      1.621333e+01
Name: log_annual_inc, dtype: float64


In [16]:
feature_cols = ['dti', 'credit_hist_length_months', 'int_rate', 'log_annual_inc', 'loan_amnt']
X = df_clean[feature_cols]
y = df_clean['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_balanced, y_train_balanced)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred_proba))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

AUC: 0.6897908285491663
Accuracy: 0.6506325625393373
              precision    recall  f1-score   support

           0       0.87      0.66      0.75    215256
           1       0.31      0.61      0.41     53255

    accuracy                           0.65    268511
   macro avg       0.59      0.63      0.58    268511
weighted avg       0.76      0.65      0.68    268511



In [17]:
df_clean.to_pickle('../data/cleaned_loans.pkl')  # overwrite with new features included

In [18]:
import numpy as np
import pandas as pd

def calculate_woe_iv(df, feature, target, bins=10):
    data = df[[feature, target]].copy()
    data['bin'] = pd.qcut(data[feature], q=bins, duplicates='drop')

    grouped = data.groupby('bin')[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()

    grouped['good_pct'] = grouped['good'] / total_good
    grouped['bad_pct'] = grouped['bad'] / total_bad

    # tiny epsilon avoids divide-by-zero if a bin has 0 goods or 0 bads
    eps = 1e-6
    grouped['woe'] = np.log((grouped['good_pct'] + eps) / (grouped['bad_pct'] + eps))
    grouped['iv'] = (grouped['good_pct'] - grouped['bad_pct']) * grouped['woe']

    total_iv = grouped['iv'].sum()
    return grouped, total_iv

In [19]:
for col in ['dti', 'credit_hist_length_months']:
    woe_table, iv_value = calculate_woe_iv(df_clean, col, 'default', bins=10)
    print(f"\n=== {col} | Total IV = {iv_value:.4f} ===")
    print(woe_table[['total', 'good', 'bad', 'woe', 'iv']])

C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])



=== dti | Total IV = 0.0740 ===
                  total    good    bad       woe        iv
bin                                                       
(-0.001, 7.28]   134466  115141  19325  0.388011  0.013349
(7.28, 10.49]    134140  113836  20304  0.327194  0.009657
(10.49, 13.02]   134442  112768  21674  0.252474  0.005902
(13.02, 15.324]  133973  110939  23034  0.175264  0.002904
(15.324, 17.61]  134306  109606  24700  0.093345  0.000847
(17.61, 19.98]   134549  108172  26377  0.014487  0.000021
(19.98, 22.59]   134185  106032  28153 -0.070655  0.000510
(22.59, 25.68]   134449  103906  30543 -0.172390  0.003131
(25.68, 29.76]   134079  100598  33481 -0.296586  0.009571
(29.76, 100.0]   133963   95281  38682 -0.495282  0.028104

=== credit_hist_length_months | Total IV = 0.0094 ===
                 total    good    bad       woe        iv
bin                                                      
(11.999, 97.0]  134857  105100  29757 -0.134894  0.001902
(97.0, 125.0]   135099  105818

C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])


In [20]:
for col in ['int_rate', 'log_annual_inc', 'loan_amnt']:
    woe_table, iv_value = calculate_woe_iv(df_clean, col, 'default', bins=10)
    print(f"\n=== {col} | Total IV = {iv_value:.4f} ===")
    print(woe_table[['total', 'good', 'bad', 'woe', 'iv']])

C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])



=== int_rate | Total IV = 0.4493 ===
                            total    good    bad       woe        iv
bin                                                                 
(5.308999999999999, 7.39]  135269  128722   6547  1.581873  0.150296
(7.39, 8.9]                135680  124223  11457  0.986720  0.071430
(8.9, 10.49]               137420  120856  16564  0.590618  0.029580
(10.49, 11.53]             134639  113909  20730  0.307072  0.008593
(11.53, 12.74]             131994  109397  22597  0.180421  0.003027
(12.74, 13.98]             131818  105320  26498 -0.016809  0.000028
(13.98, 15.22]             134427  103188  31239 -0.201856  0.004329
(15.22, 16.99]             138592  101623  36969 -0.385549  0.017125
(16.99, 19.47]             128489   87959  40530 -0.621909  0.043837
(19.47, 30.99]             134224   81082  53142 -0.974241  0.121041

=== log_annual_inc | Total IV = 0.0294 ===
                               total    good    bad       woe        iv
bin               

C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])


In [21]:
def apply_woe(df, feature, woe_table):
    bins = woe_table.index
    woe_map = {b: woe_table.loc[b, 'woe'] for b in bins}
    binned = pd.cut(df[feature], bins=[b.left for b in bins] + [bins[-1].right], include_lowest=True)
    return binned.map(lambda b: woe_map.get(b, 0))

scorecard_features = ['int_rate', 'dti', 'loan_amnt', 'log_annual_inc']  # dropped credit_hist_length_months (IV < 0.02)

woe_tables = {}
for col in scorecard_features:
    woe_tables[col], _ = calculate_woe_iv(df_clean, col, 'default', bins=10)
    df_clean[col + '_woe'] = apply_woe(df_clean, col, woe_tables[col])

print(df_clean[[c + '_woe' for c in scorecard_features]].describe())

C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby('bin')[target].agg(['count', 'sum'])
C:\Users\91703\AppData\Local\Temp\ipykernel_23884\3590051166.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future d

       log_annual_inc_woe
count        1.342552e+06
mean         3.743320e-02
std          1.550187e-01
min         -1.761626e-01
25%         -7.425120e-02
50%          2.076104e-02
75%          1.467592e-01
max          3.427819e-01


In [24]:
woe_cols = [c + '_woe' for c in scorecard_features]
df_clean[woe_cols] = df_clean[woe_cols].astype(float)   # <-- new line, forces all 4 to plain floats
X = df_clean[woe_cols]
y = df_clean['default']

In [25]:
print(df_clean[woe_cols].dtypes)

int_rate_woe          float64
dti_woe               float64
loan_amnt_woe         float64
log_annual_inc_woe    float64
dtype: object


In [26]:
df_clean.to_pickle('../data/cleaned_loans.pkl')